Colab Version of the API Integration Assignment

In [1]:
import os

folders = ["API_Integration_Assignment/config",
           "API_Integration_Assignment/providers",
           "API_Integration_Assignment/models",
           "API_Integration_Assignment/comparator",
           "API_Integration_Assignment/reporting", ]

for folder in folders:
  os.makedirs(folder, exist_ok=True)
  init_path = os.path.join(folder, "__init__.py")
  with open(init_path, "w") as f:
    f.write("")

In [2]:
config_yaml = """
locations:
  - code: DTW
    latitude: 42.2162
    longitude: -83.3554

  - code: ATL
    latitude: 33.6407
    longitude: -84.4277

weather_sources:
  - weatherapi
  - meteostat
"""

with open("API_Integration_Assignment/config/settings.yaml", "w") as f:
  f.write(config_yaml)

In [3]:
secrets_yaml = """

api_keys:
  weatherapi: "e6aa21de3e46465e818165347262206"
  meteostat: "86ef5d90aamshb092c8089e5e74ap1794f1jsnda514ac0b6a3"
"""

with open("API_Integration_Assignment/config/secrets.yaml", "w") as f:
  f.write(secrets_yaml)

In [4]:
gitignore_content = """
API_Integration_Assignment/config/secrets.yaml
__pycache__/
*.pyc
*.pyo
"""

with open(".gitignore", "w") as f:
  f.write(gitignore_content)

In [5]:
weatherCode = '''
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class WeatherInfo:
  # Normalized weather data model.
  source: str              #"weatherapi" or "meteostat"
  location: str            #"DTW" or "ATL"
  date: str                #"2026-06-21"
  avg_temp_f: Optional[float] = None   #providers don't always return every field(Meteostat doesn't return humidity)
  max_temp_f: Optional[float] = None
  min_temp_f: Optional[float] = None
  humidity: Optional[float] = None
  precipitation_in: Optional[float] = None
  wind_mph: Optional[float] = None

  def to_dict(self) -> dict:
    #dictionary format
    return {
      "source": self.source,
      "location": self.location,
      "date": self.date,
      "avg_temp_f": self.avg_temp_f,
      "max_temp_f": self.max_temp_f,
      "min_temp_f": self.min_temp_f,
      "humidity": self.humidity,
      "precipitation_in": self.precipitation_in,
      "wind_mph": self.wind_mph,
    }

  def __repr__(self):
    return (
      f"WeatherInfo(source={self.source!r}, location={self.location!r}, "
      f"date={self.date!r}, avg_temp_f={self.avg_temp_f}, "
      f"max_temp_f={self.max_temp_f}, min_temp_f={self.min_temp_f}, "
      f"humidity={self.humidity}, precipitation_in={self.precipitation_in}, "
      f"wind_mph={self.wind_mph})"
    )
'''

with open("API_Integration_Assignment/models/weatherInfo.py", "w") as f:
  f.write(weatherCode)

In [6]:
#Testing with mock hardcoded data
import sys
sys.path.insert(0, "/content")  #for Colab to find my files

from API_Integration_Assignment.models.weatherInfo import WeatherInfo

testCase = WeatherInfo(
  source="weatherapi",
  location="DTW",
  date="2026-06-21",
  avg_temp_f=72.1,
  max_temp_f=79.0,
  min_temp_f=65.2,
  humidity=68.0,
  precipitation_in=0.14,
  wind_mph=10.2
)

print("Normalized data model")
print(testCase)
print("\nAs dictionary:")
print(testCase.to_dict())

Normalized data model
WeatherInfo(source='weatherapi', location='DTW', date='2026-06-21', avg_temp_f=72.1, max_temp_f=79.0, min_temp_f=65.2, humidity=68.0, precipitation_in=0.14, wind_mph=10.2)

As dictionary:
{'source': 'weatherapi', 'location': 'DTW', 'date': '2026-06-21', 'avg_temp_f': 72.1, 'max_temp_f': 79.0, 'min_temp_f': 65.2, 'humidity': 68.0, 'precipitation_in': 0.14, 'wind_mph': 10.2}


In [7]:
!pip install requests pyyaml -q

In [8]:
configCode = '''
import yaml
import os
from datetime import date, timedelta

def load_config() -> dict:
  #Combine settings.yaml + secrets.yaml into one config dict

  base_dir = os.path.dirname(os.path.abspath(__file__))

  settings_path = os.path.join(base_dir, "settings.yaml")
  secrets_path  = os.path.join(base_dir, "secrets.yaml")

  with open(settings_path, "r") as f:
    config = yaml.safe_load(f)

  with open(secrets_path, "r") as f:
    secrets = yaml.safe_load(f)

  config["api_keys"] = secrets.get("api_keys", {})
  config["target_date"] = (date.today() - timedelta(days=1)).isoformat()

  return config
'''

with open("API_Integration_Assignment/config/configLoader.py", "w") as f:
  f.write(configCode)

In [9]:
baseWeatherProviderCode = '''
from abc import ABC, abstractmethod
from API_Integration_Assignment.models.weatherInfo import WeatherInfo

class WeatherProvider(ABC):
  #Abstract base class for all weather providers

  def __init__(self, api_key: str):
    self.api_key = api_key

  @abstractmethod
  def get_weather(self, location_code: str, latitude: float, longitude: float, date: str) -> WeatherInfo:
    """Return WeatherInfo instance
    Args:
      location_code : Airport code e.g. "DTW"
      latitude      : Location latitude
      longitude     : Location longitude
      date          : Date e.g. "2026-06-22"

    Returns:
      WeatherInfo with normalized fields
    """
    pass

  #shared unit conversion functions
  def celsius_to_fahrenheit(self, celsius: float) -> float:
    return round((celsius * 9 / 5) + 32, 2)

  def mm_to_inches(self, mm: float) -> float:
    return round(mm / 25.4, 4)

  def kph_to_mph(self, kph: float) -> float:
    return round(kph * 0.621371, 2)
'''

with open("API_Integration_Assignment/providers/baseProvider.py", "w") as f:
  f.write(baseWeatherProviderCode)

In [10]:
weatherapiProviderCode = '''
import requests
from API_Integration_Assignment.providers.baseProvider import WeatherProvider
from API_Integration_Assignment.models.weatherInfo import WeatherInfo

class WeatherApiProvider(WeatherProvider):
  #WeatherAPI returns imperial units(F, mph, inches)

  BASE_URL = "http://api.weatherapi.com/v1/history.json"

  def get_weather(self, location_code: str, latitude: float, longitude: float, date: str) -> WeatherInfo:

    params = {
      "key": self.api_key,
      "q": f"{latitude},{longitude}",
      "dt": date,
    }

    response = requests.get(self.BASE_URL, params=params, timeout=10)
    response.raise_for_status()
    data = response.json()

    #WeatherAPI response
    day = data["forecast"]["forecastday"][0]["day"]

    return WeatherInfo(
      source="weatherapi",
      location=location_code,
      date=date,
      avg_temp_f=round(day.get("avgtemp_f", 0), 2),
      max_temp_f=round(day.get("maxtemp_f", 0), 2),
      min_temp_f=round(day.get("mintemp_f", 0), 2),
      humidity=round(day.get("avghumidity", 0), 2),
      precipitation_in=round(day.get("totalprecip_in", 0), 4),
      wind_mph=round(day.get("maxwind_mph", 0), 2),
    )
'''

with open("API_Integration_Assignment/providers/weatherapiProvider.py", "w") as f:
  f.write(weatherapiProviderCode)

In [11]:
meteostatProviderCode = '''
import requests
from API_Integration_Assignment.providers.baseProvider import WeatherProvider
from API_Integration_Assignment.models.weatherInfo import WeatherInfo

class MeteostatProvider(WeatherProvider):
  #Meteostat returns metric units (C, mm, km/h)

  BASE_URL = "https://meteostat.p.rapidapi.com/point/daily"
  HOST = "meteostat.p.rapidapi.com"

  def get_weather(self, location_code: str, latitude: float, longitude: float, date: str) -> WeatherInfo:

    headers = {
      "x-rapidapi-key": self.api_key,
      "x-rapidapi-host": self.HOST,
    }

    params = {
      "lat": latitude,
      "lon": longitude,
      "start": date,
      "end": date,
    }

    response = requests.get(self.BASE_URL, headers=headers, params=params,timeout=10)
    response.raise_for_status()
    data = response.json()

    info = data.get("data", [])
    if not info:
      raise ValueError(f"Meteostat returned no data for {location_code} on {date}")

    day = info[0]

    # Meteostat params:
    # tavg=avg temp C, tmax=max temp C, tmin=min temp C
    # prcp=precipitation mm, wspd=wind speed km/h, rhum=relative humidity %

    avg_temp_f = self.celsius_to_fahrenheit(day["tavg"]) if day.get("tavg") is not None else None
    max_temp_f = self.celsius_to_fahrenheit(day["tmax"]) if day.get("tmax") is not None else None
    min_temp_f = self.celsius_to_fahrenheit(day["tmin"]) if day.get("tmin") is not None else None
    precip_in = self.mm_to_inches(day["prcp"])          if day.get("prcp") is not None else None
    wind_mph = self.kph_to_mph(day["wspd"])            if day.get("wspd") is not None else None
    humidity = day.get("rhum")

    return WeatherInfo(
      source="meteostat",
      location=location_code,
      date=date,
      avg_temp_f=avg_temp_f,
      max_temp_f=max_temp_f,
      min_temp_f=min_temp_f,
      humidity=float(humidity) if humidity is not None else None,
      precipitation_in=precip_in,
      wind_mph=wind_mph,
    )
'''

with open("API_Integration_Assignment/providers/meteostatProvider.py", "w") as f:
  f.write(meteostatProviderCode)



In [12]:
factoryWeatherProvidercode = '''
from API_Integration_Assignment.providers.weatherapiProvider import WeatherApiProvider
from API_Integration_Assignment.providers.meteostatProvider import MeteostatProvider
from API_Integration_Assignment.providers.baseProvider import WeatherProvider

PROVIDER_LIST = {
  "weatherapi": WeatherApiProvider,
  "meteostat":  MeteostatProvider,
}

def get_provider(name: str, api_key: str) -> WeatherProvider:
  #Returns the provider by config name else an error
  if name not in PROVIDER_LIST:
    available = list(PROVIDER_LIST.keys())
    raise ValueError(f"Unknown provider: {name!r}. Available: {available}")

  provider_class = PROVIDER_LIST[name]
  return provider_class(api_key=api_key)
'''

with open("API_Integration_Assignment/providers/providerFactory.py", "w") as f:
  f.write(factoryWeatherProvidercode)

print("Provider factory for new API when need to add...")

Provider factory for new API when need to add...


In [13]:
# This is testing to check end to end
import sys
sys.path.insert(0, "/content")

from API_Integration_Assignment.config.configLoader import load_config
from API_Integration_Assignment.providers.providerFactory import get_provider

# Load config and secrets
config = load_config()
target_date = config["target_date"]
locations   = config["locations"]
api_keys    = config["api_keys"]

print(f"Target date: {target_date}")
print(f"Locations: {[l['code'] for l in locations]}\n")

#location (DTW)
dtw = locations[0]

#WeatherAPI
print("Testing WeatherAPI")
weatherapi = get_provider("weatherapi", api_keys["weatherapi"])
record_weather_api  = weatherapi.get_weather(dtw["code"], dtw["latitude"], dtw["longitude"], target_date)
print(f"WeatherAPI → {record_weather_api}\n")

#Meteostat
print("Testing Meteostat")
meteostat = get_provider("meteostat", api_keys["meteostat"])
record_metos  = meteostat.get_weather(dtw["code"], dtw["latitude"], dtw["longitude"], target_date)
print(f"Meteostat  → {record_metos}")

Target date: 2026-06-23
Locations: ['DTW', 'ATL']

Testing WeatherAPI
WeatherAPI → WeatherInfo(source='weatherapi', location='DTW', date='2026-06-23', avg_temp_f=67.4, max_temp_f=78.9, min_temp_f=56.2, humidity=57, precipitation_in=0.0, wind_mph=9.6)

Testing Meteostat
Meteostat  → WeatherInfo(source='meteostat', location='DTW', date='2026-06-23', avg_temp_f=66.56, max_temp_f=74.84, min_temp_f=57.38, humidity=None, precipitation_in=0.0, wind_mph=6.28)


In [14]:
# import requests

# # Manually test the Meteostat to checkerror detail
# url = "https://meteostat.p.rapidapi.com/point/daily"

# headers = {
#     "x-rapidapi-key": api_keys["meteostat"],
#     "x-rapidapi-host": "meteostat.p.rapidapi.com",
# }

# params = {
  # "lat": 42.2162,
  # "lon": -83.3554,
  # "start": target_date,
  # "end": target_date,
# }

# response = requests.get(url, headers=headers, params=params)

# print(f"Status code : {response.status_code}")
# print(f"Response    : {response.text}")

In [15]:
comparator_code = '''
from dataclasses import dataclass, field
from typing import Optional, List
from API_Integration_Assignment.models.weatherInfo import WeatherInfo

@dataclass
class MetricDifference:

  metric: str
  source_a_value: Optional[float]
  source_b_value: Optional[float]
  difference: Optional[float]
  drift_detected: bool

@dataclass
class LocationDriftReport:
  # drift report for one location comparing two sources
  location: str
  date: str
  source_a: str
  source_b: str
  diffs: List[MetricDifference] = field(default_factory=list)
  has_drift: bool = False

class WeatherCompareEngine:
  # Compare two WeatherInfo objects

  THRESHOLDS = {
    "avg_temp_f":        2.0,   # degrees F
    "max_temp_f":        2.0,
    "min_temp_f":        2.0,
    "humidity":          5.0,   # percent
    "precipitation_in":  0.1,   # inches
    "wind_mph":          2.0,   # mph
  }

  METRIC_LABELS = {
    "avg_temp_f":       "Avg Temp (F)",
    "max_temp_f":       "Max Temp (F)",
    "min_temp_f":       "Min Temp (F)",
    "humidity":         "Humidity (%)",
    "precipitation_in": "Precipitation (in)",
    "wind_mph":         "Wind (mph)",
  }

  def compare(self, record_a: WeatherInfo, record_b: WeatherInfo) -> LocationDriftReport:
    # Compare two WeatherInfo objects

    if record_a.location != record_b.location:
      raise ValueError(
        f"Location mismatch: {record_a.location} vs {record_b.location}. "
        "Can only compare records for the same location."
      )

    report = LocationDriftReport(
      location=record_a.location,
      date=record_a.date,
      source_a=record_a.source,
      source_b=record_b.source,
    )

    for field_name, label in self.METRIC_LABELS.items():
      val_a = getattr(record_a, field_name)
      val_b = getattr(record_b, field_name)

      if val_a is None or val_b is None:
        diff = MetricDifference(
          metric=label,
          source_a_value=val_a,
          source_b_value=val_b,
          difference=None,
          drift_detected=False,
        )
      else:
        difference = round(abs(val_a - val_b), 4)
        threshold  = self.THRESHOLDS.get(field_name, 0)
        is_drift   = difference > threshold

        diff = MetricDifference(
          metric=label,
          source_a_value=val_a,
          source_b_value=val_b,
          difference=difference,
          drift_detected=is_drift,
        )

        if is_drift:
          report.has_drift = True

      report.diffs.append(diff)

    return report
'''

with open("API_Integration_Assignment/comparator/comparator.py", "w") as f:
  f.write(comparator_code)

print("Comparison done")

Comparison done


In [16]:
#testing output format
from API_Integration_Assignment.comparator.comparator import WeatherCompareEngine

comparator = WeatherCompareEngine()
dtw_report  = comparator.compare(record_weather_api, record_metos)

print(f"Location : {dtw_report.location}")
print(f"Date     : {dtw_report.date}")

print(f"{'Metric':<22} {'Source A':>10} {'Source B':>10} {'Diff':>8} {'Drift?':>8}")
print("-" * 62)

for d in dtw_report.diffs:
  val_a  = f"{d.source_a_value}" if d.source_a_value is not None else "N/A"
  val_b  = f"{d.source_b_value}" if d.source_b_value is not None else "N/A"
  diff   = f"{d.difference}"     if d.difference     is not None else "N/A"
  flag   = "YES.." if d.drift_detected else "NO!!"
  print(f"{d.metric:<22} {val_a:>10} {val_b:>10} {diff:>8} {flag:>8}")

Location : DTW
Date     : 2026-06-23
Metric                   Source A   Source B     Diff   Drift?
--------------------------------------------------------------
Avg Temp (F)                 67.4      66.56     0.84     NO!!
Max Temp (F)                 78.9      74.84     4.06    YES..
Min Temp (F)                 56.2      57.38     1.18     NO!!
Humidity (%)                   57        N/A      N/A     NO!!
Precipitation (in)            0.0        0.0      0.0     NO!!
Wind (mph)                    9.6       6.28     3.32    YES..


In [17]:
reporter_code = '''
import json
import csv
import os
from datetime import datetime
from typing import List
from API_Integration_Assignment.comparator.comparator import LocationDriftReport, MetricDifference

class DriftReportClass:
  def __init__(self, output_dir: str = "reports"):
    self.output_dir = output_dir
    os.makedirs(output_dir, exist_ok=True)

  def print_console_report(self, reports: List[LocationDriftReport]) -> None:
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print("=" * 55)
    print("               WEATHER DRIFT REPORT   ")
    print("=" * 55)

    for report in reports:
      self._print_location_summary(report)
      self._print_location_table(report)

    total   = len(reports)
    drifted = sum(1 for r in reports if r.has_drift)
    clean   = total - drifted
    # print("=" * 55)
    # print("  SUMMARY")
    # print(f"  Locations checked : {total}")
    # print(f"  Drift detected    : {drifted}")
    # print(f"  Clean             : {clean}")
    # print("=" * 55)

  def _print_location_summary(self, report: LocationDriftReport) -> None:
    print()
    print(f"  Location = {report.location}")

    for d in report.diffs:
      if d.difference is None:
        if "Humidity" in d.metric:
          #Meteostat's free tier does not return humidity for all locations
          print(f"  Humidity Difference = N/A (Meteostat did not return humidity)")
        continue

      if "Avg" in d.metric:
          print(f"  Temperature Difference = {d.difference}°F")
      elif "Humidity" in d.metric:
          print(f"  Humidity Difference = {d.difference}%")
      elif "Wind" in d.metric:
          print(f"  Wind Difference = {d.difference} mph")
      elif "Precipitation" in d.metric:
          print(f"  Precipitation Difference = {d.difference} in")

    print()

  def _print_location_table(self, report: LocationDriftReport) -> None:
    print("=" * 55)
    print(f"Location: {report.location}")
    print(f"Date: {report.date}")
    print()
    print(f"{'Metric':<22} {'Source A':>10} {'Source B':>10} {'Diff':>8}")
    print("-" * 55)

    for d in report.diffs:
      val_a = f"{d.source_a_value}" if d.source_a_value is not None else "N/A"
      val_b = f"{d.source_b_value}" if d.source_b_value is not None else "N/A"
      diff  = f"{d.difference}"     if d.difference     is not None else "N/A"
      print(f"{d.metric:<22} {val_a:>10} {val_b:>10} {diff:>8}")

    print()
    status = "DRIFT DETECTED" if report.has_drift else "NO DRIFT"
    print(f"Status: {status}")
    print("=" * 55)

  def save_json_report(self, reports: List[LocationDriftReport]) -> str:
    output = {
      "generated_at": datetime.now().isoformat(),
      "total_locations": len(reports),
      "drift_detected_count": sum(1 for r in reports if r.has_drift),
      "reports": []
    }
    for report in reports:
      report_dict = {
        "location": report.location,
        "date": report.date,
        "source_a": report.source_a,
        "source_b": report.source_b,
        "status": "DRIFT DETECTED" if report.has_drift else "NO DRIFT",
        "metrics": []
      }
      for d in report.diffs:
        report_dict["metrics"].append({
          "metric": d.metric,
          "source_a_value": d.source_a_value,
          "source_b_value": d.source_b_value,
          "difference": d.difference,
          "drift_detected": d.drift_detected,
        })
      output["reports"].append(report_dict)

    filepath = os.path.join(self.output_dir, "drift_report.json")
    with open(filepath, "w") as f:
      json.dump(output, f, indent=2)
    print(f"JSON report saved → {filepath}")
    return filepath

  def save_csv_report(self, reports: List[LocationDriftReport]) -> str:
    filepath = os.path.join(self.output_dir, "drift_report.csv")
    fieldnames = [
      "location", "date", "source_a", "source_b",
      "metric", "source_a_value", "source_b_value",
      "difference", "drift_detected", "location_status"
    ]
    with open(filepath, "w", newline="") as f:
      writer = csv.DictWriter(f, fieldnames=fieldnames)
      writer.writeheader()
      for report in reports:
        location_status = "DRIFT DETECTED" if report.has_drift else "NO DRIFT"
        for d in report.diffs:
          writer.writerow({
            "location":        report.location,
            "date":            report.date,
            "source_a":        report.source_a,
            "source_b":        report.source_b,
            "metric":          d.metric,
            "source_a_value":  d.source_a_value,
            "source_b_value":  d.source_b_value,
            "difference":      d.difference,
            "drift_detected":  d.drift_detected,
            "location_status": location_status,
          })
    print(f"CSV report saved → {filepath}")
    return filepath
'''

with open("API_Integration_Assignment/reporting/reporter.py", "w") as f:
  f.write(reporter_code)

In [18]:
#testing final report format
from importlib import reload
import API_Integration_Assignment.reporting.reporter as r_module
reload(r_module)
from API_Integration_Assignment.reporting.reporter import DriftReportClass

reporter = DriftReportClass(output_dir="reports")
reporter.print_console_report([dtw_report])

               WEATHER DRIFT REPORT   

  Location = DTW
  Temperature Difference = 0.84°F
  Humidity Difference = N/A (Meteostat did not return humidity)
  Precipitation Difference = 0.0 in
  Wind Difference = 3.32 mph

Location: DTW
Date: 2026-06-23

Metric                   Source A   Source B     Diff
-------------------------------------------------------
Avg Temp (F)                 67.4      66.56     0.84
Max Temp (F)                 78.9      74.84     4.06
Min Temp (F)                 56.2      57.38     1.18
Humidity (%)                   57        N/A      N/A
Precipitation (in)            0.0        0.0      0.0
Wind (mph)                    9.6       6.28     3.32

Status: DRIFT DETECTED


# **Main runner code:**

In [19]:
main_runner_code = '''
import sys
import os
sys.path.insert(0, "/content")

from API_Integration_Assignment.config.configLoader import load_config
from API_Integration_Assignment.providers.provider_factory import get_provider
from API_Integration_Assignment.comparator.comparator import WeatherCompareEngine
from API_Integration_Assignment.reporting.reporter import DriftReportClass

def run():
    print("Weather Drift Report\\n")

    #1. Load config
    config      = load_config()
    locations   = config["locations"]
    sources     = config["weather_sources"]
    api_keys    = config["api_keys"]
    target_date = config["target_date"]

    print(f"Date      : {target_date}")
    print(f"Locations : {[l[\'code\'] for l in locations]}")
    print(f"Sources   : {sources}\\n")

    # 2. Fetch weather from all providers for all location
    comparator = WeatherCompareEngine()
    reporter   = DriftReportClass(output_dir="reports")
    all_reports = []

    for location in locations:
        code = location["code"]
        lat  = location["latitude"]
        lon  = location["longitude"]

        print(f"Fetching data for {code}")

        records = {}
        for source in sources:
            try:
                provider = get_provider(source, api_keys[source])
                record   = provider.get_weather(code, lat, lon, target_date)
                records[source] = record
                print(f"  {source:<15} → avg={record.avg_temp_f}F  "
                      f"max={record.max_temp_f}F  min={record.min_temp_f}F  "
                      f"humidity={record.humidity}%  "
                      f"precip={record.precipitation_in}in  "
                      f"wind={record.wind_mph}mph")
            except Exception as e:
                print(f"  {source:<15} → ERROR: {e}")
                records[source] = None

        # 3. Compare the two sourcees
        source_a, source_b = sources[0], sources[1]
        rec_a = records.get(source_a)
        rec_b = records.get(source_b)

        if rec_a and rec_b:
            drift_report = comparator.compare(rec_a, rec_b)
            all_reports.append(drift_report)
        else:
            print(f" Skipping comparison for {code} — missing data from one source\\n")

        print()

    # 4. Generate reports
    if all_reports:
        reporter.print_console_report(all_reports)
        reporter.save_json_report(all_reports)
        reporter.save_csv_report(all_reports)
        print("\\nDONE. Reports saved under reports/")
    else:
        print("No reports generated — check API errors above.")

if __name__ == "__main__":
    run()
'''

with open("API_Integration_Assignment/main.py", "w") as f:
    f.write(main_runner_code)

print("Main code!")

Main code!
